# API Tools

For this section we'll just get a default ReAct agent to run our tools; this will let us focus our debugging in the tools themselves rather than the agent.

In [2]:
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

**EDIT: Moved all API handling code to [`football_api_utils.py`](football_api_utils.py)** so it can be used in other notebooks as well.

In [ ]:
!uv pip install python-dotenv requests pydantic

# Import the Football API utility functions from our module
from football_api_utils import (
    Response, 
    ValidResponse, 
    ErrorResponse, 
    check_api_response_status, 
    call_football_api
)

Audited 2 packages in 28ms
Audited 1 package in 5ms
Audited 1 package in 5ms


In [33]:
# Test the API functions
print("Testing API status...")
status_response = call_football_api("GET", "status")
print(f"Status response: {status_response}")

print("\nTesting error handling with invalid endpoint...")
error_response = call_football_api("GET", "invalid_endpoint")
print(f"Error response: {error_response}")

Testing API status...
Status response: data={'get': 'status', 'parameters': [], 'errors': [], 'results': 0, 'paging': {'current': 1, 'total': 1}, 'response': {'account': {'firstname': 'Vasco', 'lastname': 'Peleteiro', 'email': 'vasco@augustalabs.ai'}, 'subscription': {'plan': 'Pro', 'end': '2025-07-05T20:58:08+00:00', 'active': True}, 'requests': {'current': 30, 'limit_day': 7500}}}

Testing error handling with invalid endpoint...
Error response: data={'get': 'invalid_endpoint', 'parameters': [], 'errors': {'endpoint': 'The Invalid_endpoint endpoint does not exist.'}, 'results': 0, 'paging': {'current': 1, 'total': 1}, 'response': []}
Status response: data={'get': 'status', 'parameters': [], 'errors': [], 'results': 0, 'paging': {'current': 1, 'total': 1}, 'response': {'account': {'firstname': 'Vasco', 'lastname': 'Peleteiro', 'email': 'vasco@augustalabs.ai'}, 'subscription': {'plan': 'Pro', 'end': '2025-07-05T20:58:08+00:00', 'active': True}, 'requests': {'current': 30, 'limit_day': 7

We need to implement a tool that can query the [Football API](https://www.api-football.com/documentation-v3) to get information about football leagues, teams, players, and matches.

1. Classificações de equipas — <https://www.api-football.com/documentation-v3#tag/Standings/operation/get-standings>
2. Próximos jogos — <https://www.api-football.com/documentation-v3#tag/Fixtures/operation/get-fixtures> see `next` parameter
3. Últimos jogos — <https://www.api-football.com/documentation-v3#tag/Fixtures/operation/get-fixtures> see `last` parameter
4. Jogos específicos (ex: SLB vs SCP para a liga em 2012/13) — we need to get the fixture ID first, then use it to get the match details.
5. Resultados de jogos específicos
6. Eventos de jogos específicos (ex.: golos, cartões, substituições).
7. Estatísticas de jogadores (ex: número de golos do jogador X na época Y)

Qualquer equipa, jogo ou jogador das top 7 ligas europeias + das 3 competições europeias.

Extra:

8. Odds
9. H2H
10. ...


## API Football

There doesn't seem to be a OpenAPI spec for this API, but we'll use the documentation to implement the tools we need and see how it goes.

### 0. Setup

**EDIT: Moved cache generation (`top_leagues.json` and `top_teams.json`)  to [`03a_tools_setup.ipynb`](03a_tools_setup.ipynb) to avoid clutter and repeated execution of costly API calls.**

In [11]:
import json

# Define function to get league ID by name
def get_league_id_by_name(league_name: str) -> int | None:
    """
    Get the league ID by its name.

    Args:
        league_name: Name of the league

    Returns:
        League ID if found, otherwise None
    """
    try:
        leagues = json.load(open("top_leagues.json"))
    except FileNotFoundError:
        logger.warning("top_leagues.json not found. Will fetch from API...")
        leagues = {}

    # Exact match first
    if league_name in leagues:
        return leagues[league_name]

    # If the league is not found, partial match the name
    #for league_key in leagues:
    #    if league_name.lower() in league_key.lower() or league_key.lower() in league_name.lower():
    #        return leagues[league_key]

    # If no match is found, fetch it from the API:
    logger.warning(
        f"League '{league_name}' not found in selected leagues. Fetching from API..."
    )
    response = call_football_api("GET", "leagues", params={"search": league_name})
    if isinstance(response, ValidResponse):
        leagues_parsed = extract_league_info(response.data)
        for league in leagues_parsed:
            if league["name"].lower() in league_name.lower() or league_name.lower() in league["name"].lower():
                return league["id"]
        logger.error(f"League '{league_name}' not found in API response.")

    return None


def list_leagues() -> list[str]:
    """
    List all available leagues.

    Returns:
        List of league names
    """
    try:
        leagues = json.load(open("top_leagues.json"))
        return list(leagues.keys())
    except FileNotFoundError:
        logger.warning("top_leagues.json not found.")
        return []

In [12]:
# Test exact league match
league_name = "Premier League"
league_id = get_league_id_by_name(league_name)
print(f"League: {league_name}, ID: {league_id}")

League: Premier League, ID: 39


In [13]:
# Test partial match
league_name_partial = "Premier"
league_id_partial = get_league_id_by_name(league_name_partial)
print(f"League (partial): {league_name_partial}, ID: {league_id_partial}")

League (partial): Premier, ID: 39


In [14]:
# Test list leagues function
available_leagues = list_leagues()
print(f"Available leagues: {available_leagues[:5]}...")  # Show first 5

Available leagues: ['Ligue 1', 'Premier League', 'Bundesliga', 'Serie A', 'Eredivisie']...


In [15]:
# Test with a league not in the list
league_name_not_found = "Nonexistent League"
league_id_not_found = get_league_id_by_name(league_name_not_found)
print(f"League (not found): {league_name_not_found}, ID: {league_id_not_found}")

ERROR:__main__:League 'Nonexistent League' not found in API response.
ERROR:__main__:League 'Nonexistent League' not found in API response.


League (not found): Nonexistent League, ID: None


In [16]:
# Test with a league not in the list that exists in the API
league_name_exists = "Jupiler Pro League"
league_id_exists = get_league_id_by_name(league_name_exists)
print(f"League (exists in API): {league_name_exists}, ID: {league_id_exists}")

League (exists in API): Jupiler Pro League, ID: 144


In [18]:
def get_team_id_by_name(team_name: str) -> int | None:
    """
    Get the team ID by its name.

    Args:
        team_name: Name of the team

    Returns:
        Team ID if found, otherwise None
    """
    try:
        teams = json.load(open("top_teams.json"))
    except FileNotFoundError:
        logger.warning("top_teams.json not found. Will fetch from API...")
        teams = {}

    # Exact match first
    if team_name in teams:
        return teams[team_name]

    # If the team is not found, try partial match
    for team_key, team_id in teams.items():
        if team_name.lower() in team_key.lower() or team_key.lower() in team_name.lower():
            return team_id

    # If no match is found, fetch it from the API:
    logger.warning(
        f"Team '{team_name}' not found in cached teams. Fetching from API..."
    )
    response = call_football_api("GET", "teams", params={"search": team_name})

    if isinstance(response, ValidResponse):
        for team_data in response.data["response"]:
            team = team_data["team"]
            if team["name"].lower() == team_name.lower():
                return team["id"]

    return None


def list_teams() -> list[str]:
    """
    List all available teams from cached data.

    Returns:
        List of team names
    """
    try:
        teams = json.load(open("top_teams.json"))
        return list(teams.keys())
    except FileNotFoundError:
        logger.warning("top_teams.json not found.")
        return []



In [19]:
# Test exact team name match
team_name = "Manchester United"
team_id = get_team_id_by_name(team_name)
print(f"Team: {team_name}, ID: {team_id}")

Team: Manchester United, ID: 33


In [20]:
# Test partial team name match
team_name_partial = "Arsenal"
team_id_partial = get_team_id_by_name(team_name_partial)
print(f"Team (partial): {team_name_partial}, ID: {team_id_partial}")

Team (partial): Arsenal, ID: 42


In [21]:
# Test with a team not in the list
team_name_not_found = "Nonexistent Team"
team_id_not_found = get_team_id_by_name(team_name_not_found)
print(f"Team (not found): {team_name_not_found}, ID: {team_id_not_found}")

Team (not found): Nonexistent Team, ID: None


In [22]:
# Test with a team that exists in the API but not in the cached data
team_name_exists = "Fluminense"
team_id_exists = get_team_id_by_name(team_name_exists)
print(f"Team (exists in API): {team_name_exists}, ID: {team_id_exists}")

Team (exists in API): Fluminense, ID: 124


In [23]:
# Test list teams function
available_teams = list_teams()
print(f"Total teams available: {len(available_teams)}")
print(f"First 5 teams: {available_teams[:5]}")  # Show first 5

Total teams available: 1622
First 5 teams: ['Angers', 'Lille', 'Lyon', 'Marseille', 'Montpellier']


## 1. Team standings

<https://www.api-football.com/documentation-v3#tag/Standings/operation/get-standings>

In [24]:
def get_standings(league_name: str, season: int, team_name: str | None = None) -> Response:
    """
    Get the standings for a league or specific team in a league.
    
    Args:
        league_name: Name of the league (e.g., "Premier League", "La Liga")
        season: Season year (4 digits, e.g., 2024)
        team_name: Optional team name to filter standings for specific team
        
    Returns:
        ValidResponse with standings data or ErrorResponse with error details
    """
    # Get league ID from the league name
    league_id = get_league_id_by_name(league_name)
    if league_id is None:
        logger.error(f"League '{league_name}' not found")
        return ErrorResponse(error=f"League '{league_name}' not found")
    
    # Prepare parameters for the API call
    params = {
        "league": league_id,
        "season": season
    }
    
    # If team name is provided, get team ID and add to params
    if team_name:
        team_id = get_team_id_by_name(team_name)
        if team_id is None:
            logger.error(f"Team '{team_name}' not found")
            return ErrorResponse(error=f"Team '{team_name}' not found")
        params["team"] = team_id
    
    logger.info(f"Fetching standings for {league_name} (ID: {league_id}) season {season}")
    if team_name:
        logger.info(f"Filtering for team: {team_name}")
    
    # Make the API call
    try:
        response = call_football_api("GET", "standings", params=params)
        
        if isinstance(response, ValidResponse) and "response" in response.data and response.data["response"]:
            logger.info(f"Successfully retrieved standings data")
            return response
        elif isinstance(response, ValidResponse):
            logger.warning(f"No standings data found for {league_name} season {season}")
            return ErrorResponse(error=f"No standings data found for {league_name} season {season}")
        else:
            return response  # Already an ErrorResponse
            
    except Exception as e:
        logger.error(f"Error fetching standings: {str(e)}")
        return ErrorResponse(error=f"Error fetching standings: {str(e)}")


def format_standings_table(standings_response: Response) -> str:
    """
    Format the standings response into a readable table string.
    
    Args:
        standings_response: Response from get_standings function
        
    Returns:
        Formatted string table of the standings
    """
    if not isinstance(standings_response, ValidResponse):
        return f"Error: {standings_response.error}" # type: ignore
    
    data = standings_response.data
    if "response" not in data or not data["response"]:
        return "No standings data available"
    
    # Extract standings data
    standings_data = data["response"][0]["league"]["standings"][0]
    
    # Create table header
    table = f"{'Pos':<4} {'Team':<25} {'GP':<3} {'W':<3} {'D':<3} {'L':<3} {'GF':<3} {'GA':<3} {'GD':<4} {'Pts':<4}\n"
    table += "-" * 75 + "\n"
    
    # Add each team's data
    for team_data in standings_data:
        rank = team_data["rank"]
        team_name = team_data["team"]["name"]
        all_stats = team_data["all"]
        
        played = all_stats["played"]
        wins = all_stats["win"]
        draws = all_stats["draw"]
        losses = all_stats["lose"]
        goals_for = all_stats["goals"]["for"]
        goals_against = all_stats["goals"]["against"]
        goal_diff = goals_for - goals_against
        points = team_data["points"]
        
        # Truncate team name if too long
        display_name = team_name[:24] if len(team_name) > 24 else team_name
        
        table += f"{rank:<4} {display_name:<25} {played:<3} {wins:<3} {draws:<3} {losses:<3} {goals_for:<3} {goals_against:<3} {goal_diff:<4} {points:<4}\n"
    
    return table


First we test the function:

In [25]:
test_standings = get_standings("Premier League", 2024)
print("Raw API Response:")
print(test_standings)
print("\n" + "="*50 + "\n")
print("Formatted Table:")
print(format_standings_table(test_standings))

INFO:__main__:Fetching standings for Premier League (ID: 39) season 2024
INFO:__main__:Successfully retrieved standings data
INFO:__main__:Successfully retrieved standings data


Raw API Response:
data={'get': 'standings', 'parameters': {'league': '39', 'season': '2024'}, 'errors': [], 'results': 1, 'paging': {'current': 1, 'total': 1}, 'response': [{'league': {'id': 39, 'name': 'Premier League', 'country': 'England', 'logo': 'https://media.api-sports.io/football/leagues/39.png', 'flag': 'https://media.api-sports.io/flags/gb-eng.svg', 'season': 2024, 'standings': [[{'rank': 1, 'team': {'id': 40, 'name': 'Liverpool', 'logo': 'https://media.api-sports.io/football/teams/40.png'}, 'points': 84, 'goalsDiff': 45, 'group': 'Premier League', 'form': 'DLDLW', 'status': 'same', 'description': 'Champions League', 'all': {'played': 38, 'win': 25, 'draw': 9, 'lose': 4, 'goals': {'for': 86, 'against': 41}}, 'home': {'played': 19, 'win': 14, 'draw': 4, 'lose': 1, 'goals': {'for': 42, 'against': 16}}, 'away': {'played': 19, 'win': 11, 'draw': 5, 'lose': 3, 'goals': {'for': 44, 'against': 25}}, 'update': '2025-05-26T00:00:00+00:00'}, {'rank': 2, 'team': {'id': 42, 'name': 'Arse

Then we convert it into a tool and create a ReAct agent to use it.

In [26]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

class GetStandingsInput(BaseModel):
    league_name: str = Field(..., description="Name of the league (e.g., 'Premier League', 'La Liga')")
    season: int = Field(..., description="Season year (4 digits, e.g., 2024)")
    team_name: Optional[str] = Field(None, description="Optional team name to filter standings for specific team")

# create a tool from the get_standings function
get_standings_tool = StructuredTool.from_function(
    get_standings,
    name="get_standings",
    description="Get the standings for a league or specific team in a league.",
    args_schema=GetStandingsInput,
    return_direct=False
)

print(get_standings_tool.name)
print(get_standings_tool.description)
print(get_standings_tool.args)

get_standings
Get the standings for a league or specific team in a league.
{'league_name': {'description': "Name of the league (e.g., 'Premier League', 'La Liga')", 'title': 'League Name', 'type': 'string'}, 'season': {'description': 'Season year (4 digits, e.g., 2024)', 'title': 'Season', 'type': 'integer'}, 'team_name': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Optional team name to filter standings for specific team', 'title': 'Team Name'}}


In [27]:
!uv pip install -qU "langchain[google-genai]"

import getpass
import os
from langchain_google_genai import ChatGoogleGenerativeAI

if not os.environ.get("GOOGLE_API_KEY"):
  os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

model = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

In [28]:
# Create a ReAct agent to use the tool
from langgraph.prebuilt import create_react_agent

react_agent = create_react_agent(
    model=model,
    tools=[get_standings_tool],
    prompt="You are a helpful assistant. The latest season is 2024 and we are in early 2025."
)

Let's test our agent with a simple query:

In [29]:
react_agent.invoke(
    {"messages": [{"role": "user", "content": "what is the current standing of Sporting in the Primeira Liga?"}]}
)

INFO:__main__:Fetching standings for Primeira Liga (ID: 94) season 2024
INFO:__main__:Filtering for team: Sporting
INFO:__main__:Filtering for team: Sporting
INFO:__main__:Successfully retrieved standings data
INFO:__main__:Successfully retrieved standings data


{'messages': [HumanMessage(content='what is the current standing of Sporting in the Primeira Liga?', additional_kwargs={}, response_metadata={}, id='fd6835ca-8c81-400a-a2be-711427adf8a8'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_standings', 'arguments': '{"league_name": "Primeira Liga", "season": 2024.0, "team_name": "Sporting"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--6b0c7e60-0044-4f95-be91-ac3450fccf61-0', tool_calls=[{'name': 'get_standings', 'args': {'league_name': 'Primeira Liga', 'season': 2024.0, 'team_name': 'Sporting'}, 'id': 'f2ab9ae9-3d78-4694-842c-8f8e8bc6d16b', 'type': 'tool_call'}], usage_metadata={'input_tokens': 114, 'output_tokens': 16, 'total_tokens': 130, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content="data={'get': 'standings', 'parameters': {'league': '94', 'season': '2024', 'team

And now a fake query to see how it handles errors:

In [30]:
react_agent.invoke(
    {"messages": [{"role": "user", "content": "what is the current standing of Sporting in the Eredivisie?"}]}
)

INFO:__main__:Fetching standings for Eredivisie (ID: 88) season 2024
INFO:__main__:Filtering for team: Sporting
INFO:__main__:Filtering for team: Sporting


{'messages': [HumanMessage(content='what is the current standing of Sporting in the Eredivisie?', additional_kwargs={}, response_metadata={}, id='33f589bc-9063-42be-800e-d6b7de43890f'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_standings', 'arguments': '{"league_name": "Eredivisie", "season": 2024.0, "team_name": "Sporting"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--bacd1c1e-67ab-4adf-8614-2f492d8af1f9-0', tool_calls=[{'name': 'get_standings', 'args': {'league_name': 'Eredivisie', 'season': 2024.0, 'team_name': 'Sporting'}, 'id': '4a0c7b62-ccdf-42d4-8c9d-19c5b6130526', 'type': 'tool_call'}], usage_metadata={'input_tokens': 114, 'output_tokens': 16, 'total_tokens': 130, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content="error='No standings data found for Eredivisie season 2024' status_code=None", name='get_s

## 2. Upcoming fixtures

<https://www.api-football.com/documentation-v3#tag/Fixtures/operation/get-fixtures> see `next` parameter

## 3. Last fixtures

<https://www.api-football.com/documentation-v3#tag/Fixtures/operation/get-fixtures> see `last` parameter

## 4. Specific fixtures

## 5. Match results

## 6. Match events

## 7. Player statistics

## Extra

### 8. Odds

### 9. Head-to-head (H2H)